# Data preparation

**Goal:** prepare HotpotQA distractor examples for retrieval-augmented QA experiments.

We create two tables:

1. `questions.parquet`: one row per question.
2. `corpus.parquet`: one row per paragraph/document chunk.

For this case study, a paragraph is treated as one retrieval document. HotpotQA distractor examples already contain paragraph-level candidates and sentence-level supporting facts.

We use HotpotQA because it provides not only gold answers, but also supporting facts. This lets us evaluate two separate components:

- **retrieval quality**: whether the external index returns supporting paragraphs;
- **generation quality**: whether the model produces the correct answer when conditioned on retrieved evidence.

In [1]:
%pip install -r ../requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/polinakorobeinikova/IU/NLP/case-study/case-study-rag-factual-consistency


In [3]:
import json
import random
from datetime import datetime

import pandas as pd
from datasets import load_dataset

from src.config import (
    DATASET_NAME, DATASET_CONFIG, DATASET_SPLIT,
    PROCESSED_DIR, RANDOM_SEED
)
from src.data_utils import build_tables, save_manifest

random.seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 160)

## Configuration

In [4]:
N_EXAMPLES = 5000

dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=f"{DATASET_SPLIT}[:{N_EXAMPLES}]"
)

dataset

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 5000
})

In [5]:
# Inspect one raw example
example = dataset[0]
example.keys(), example["question"], example["answer"]

(dict_keys(['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']),
 'Were Scott Derrickson and Ed Wood of the same nationality?',
 'yes')

In [6]:
questions_df, corpus_df = build_tables(dataset)

print("Questions:", questions_df.shape)
print("Corpus:", corpus_df.shape)

display(questions_df.head(3))
display(corpus_df.head(3))

Questions: (5000, 7)
Corpus: (49774, 8)


,sample_id,question,answer,type,level,support_titles,gold_doc_ids
0,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same nationality?,yes,comparison,hard,"[""Ed Wood"", ""Scott Derrickson""]","[""5a8b57f25542995d1e6f1371::1::Scott Derrickson"", ""5a8b57f25542995d1e6f1371::4::Ed Wood""]"
1,5a8c7595554299585d9e36b6,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,bridge,hard,"[""Kiss and Tell (1945 film)"", ""Shirley Temple""]","[""5a8c7595554299585d9e36b6::1::Shirley Temple"", ""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)""]"
2,5a85ea095542994775f606a8,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,bridge,hard,"[""Animorphs"", ""The Hork-Bajir Chronicles""]","[""5a85ea095542994775f606a8::2::The Hork-Bajir Chronicles"", ""5a85ea095542994775f606a8::8::Animorphs""]"


,doc_id,sample_id,title,paragraph_idx,text,sentences_json,support_sentence_ids,is_supporting_doc
0,5a8b57f25542995d1e6f1371::0::Ed Wood (film),5a8b57f25542995d1e6f1371,Ed Wood (film),0,"Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood. T...","[""Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.""...",[],False
1,5a8b57f25542995d1e6f1371::1::Scott Derrickson,5a8b57f25542995d1e6f1371,Scott Derrickson,1,"Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer. He lives in Los Angeles, California. He is best known for direct...","[""Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer."", "" He lives in Los Angeles, California."", "" He is best known fo...",[0],True
2,"5a8b57f25542995d1e6f1371::2::Woodson, Arkansas",5a8b57f25542995d1e6f1371,"Woodson, Arkansas",2,"Woodson is a census-designated place (CDP) in Pulaski County, Arkansas, in the United States. Its population was 403 at the 2010 census. It is part of the...","[""Woodson is a census-designated place (CDP) in Pulaski County, Arkansas, in the United States."", "" Its population was 403 at the 2010 census."", "" It is par...",[],False


## Sanity checks

Each HotpotQA distractor example should have around 10 paragraphs. Usually, 2 titles are supporting documents.

In [7]:
paragraphs_per_question = corpus_df.groupby("sample_id").size()
support_docs_per_question = corpus_df[corpus_df["is_supporting_doc"]].groupby("sample_id").size()

print(paragraphs_per_question.describe())
print(support_docs_per_question.describe())

assert questions_df["sample_id"].is_unique
assert corpus_df["doc_id"].is_unique
assert len(corpus_df) > len(questions_df)
assert corpus_df["is_supporting_doc"].sum() > 0

count    5000.000000
mean        9.954800
std         0.549925
min         2.000000
25%        10.000000
50%        10.000000
75%        10.000000
max        10.000000
dtype: float64
count    5000.0
mean        2.0
std         0.0
min         2.0
25%         2.0
50%         2.0
75%         2.0
max         2.0
dtype: float64


In [8]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

questions_path = PROCESSED_DIR / "questions.parquet"
corpus_path = PROCESSED_DIR / "corpus.parquet"
manifest_path = PROCESSED_DIR / "manifest.json"

questions_df.to_parquet(questions_path, index=False)
corpus_df.to_parquet(corpus_path, index=False)

manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_config": DATASET_CONFIG,
    "dataset_split": DATASET_SPLIT,
    "n_examples": len(questions_df),
    "n_documents": len(corpus_df),
    "document_unit": "HotpotQA paragraph",
    "created_at": datetime.now().isoformat() + "Z",
    "questions_path": str(questions_path),
    "corpus_path": str(corpus_path),
}

save_manifest(manifest_path, manifest)

manifest

{'dataset_name': 'hotpotqa/hotpot_qa',
 'dataset_config': 'distractor',
 'dataset_split': 'validation',
 'n_examples': 5000,
 'n_documents': 49774,
 'document_unit': 'HotpotQA paragraph',
 'created_at': '2026-04-27T16:20:27.424024Z',
 'questions_path': '/Users/polinakorobeinikova/IU/NLP/case-study/case-study-rag-factual-consistency/data/processed/questions.parquet',
 'corpus_path': '/Users/polinakorobeinikova/IU/NLP/case-study/case-study-rag-factual-consistency/data/processed/corpus.parquet'}